In [1]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/nse_security_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today
today = datetime.today()

file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"nse_security_{file_date}.csv"
)

try:

    url = (
        "https://nsearchives.nseindia.com/"
        "content/equities/EQUITY_L.csv"
    )

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    df = pd.read_csv(
        io.BytesIO(response.content)
    )

    # keep only equity series
    df = df[
        df[" SERIES"] == "EQ"
    ]

    # clean column names
    df.columns = (
        df.columns
        .str.strip()
    )

    # save snapshot
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)
    message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# logging
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "nse_security",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, 793d20da-d450-48d2-b3bb-f842d413a9ce, 3, Finished, Available, Finished, False)

SUCCESS
Rows: 2121
File saved
